# Atlas-5K 25M: the 10M model grown again, and the widened head-to-head (ParadoxAI Lab)

Rules and predictions: `atlas/README.md`, "Atlas-5K 25M", written before this notebook ran.

**Run:** Runtime → Change runtime type → **A100** (L4 also works, slower) → Save → **Run all**, then allow Google Drive
(company account). Training (about 3.5 h on an A100) resumes from Drive after a disconnect: open the notebook and Run all
again. The scoring cells only run once training has finished.


In [ ]:
# 1. Drive (needed: the 10M checkpoint, the opponents' saved scores and the dataset copy live there) and the GPU
import os, shutil, json, glob, subprocess
from google.colab import drive
drive.mount('/content/drive', timeout_ms=300000)
W = '/content/drive/MyDrive/PWM-10M'; H2 = f'{W}/small'
A10 = f'{W}/out/w5k_atlas_10m_h24_g1_s600.pt'
TAG = 'w5k_atlas_25m_h24_g2'; A25 = f'{W}/out/{TAG}_s600.pt'; RES25 = f'{W}/out/{TAG}_s600.json'
assert os.path.exists(A10), 'the 10M checkpoint is missing from Drive'
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
# 2. Code (public repository)
!rm -rf /content/pwm && git clone -q --depth 1 -b atlas-v0.1 https://github.com/paradoxailab/paradox-world-model.git /content/pwm
!cd /content/pwm && git log --oneline -1


In [ ]:
# 3. WEATHER-5K from the Drive copy
R = '/content/raw'; os.makedirs(R, exist_ok=True)
if not os.path.exists(f'{R}/W5K_X.npy'):
    shutil.copy(f'{W}/data/W5K_META.npz', R); shutil.copy(f'{W}/data/W5K_X.npy', R)
!ls -la /content/raw


In [ ]:
# 4. Train the 25M model: the adopted 10M grown (hidden width x2.5 in every block, one encoder block added; same
#    forecasts at the start). Check G0: the 'step 0' validation must be about 0.2741 (the 10M's best on the same stations).
%cd /content/pwm/atlas/delhi
os.environ['PW_RAW'] = '/content/raw'; os.environ['ATLAS_OUT'] = f'{W}/out'
if not os.path.exists(RES25):
    !python -u run_atlas_5k.py --init {A10} --grow 1,8,7,5 --max-params 25000000 --k 32 --steps 80000 --lr 2e-4 --patience 8 --tag {TAG} --test-if-below 0.27316 --pause-at-cap --cap 660 2>&1 | grep --line-buffered -v "Warning\|warn(" | tee -a "{W}/out/log_{TAG}.txt"
else:
    print('25M training already finished:', RES25)
DONE = os.path.exists(RES25)
print('TRAINING FINISHED' if DONE else 'TRAINING NOT FINISHED (paused): open the notebook again and Run all to resume')


In [ ]:
# 5. Atlas 25M on the head-to-head windows: 2022 validation and the 2023 test (every 8th station, every 2nd day)
SC = 'python -u score_tsfm_w5k.py --stations-every 8'
if DONE:
    for yr in ('val', 'test'):
        !ATLAS_OUT={H2} PW_RAW=/content/raw {SC} --year {yr} --origin-every 48 --atlas atlas25m={A25} --models "" 2>&1 | grep --line-buffered -v Warning


In [ ]:
# 6. Live period: rebuild exactly the Phase 5b data (NOAA ISD-Lite, 1 Jan 2024 to 24 Aug 2025 22:00 UTC) and check it is
#    identical to what the opponents were scored on: the 10M model re-scored must give the same per-window errors.
LIVE = '/content/live'
if DONE:
    if os.path.exists(f'{W}/live/W5K_X.npy'):
        os.makedirs(LIVE, exist_ok=True)
        for f in ('W5K_X.npy', 'W5K_OBS.npy', 'W5K_META.npz'): shutil.copy(f'{W}/live/{f}', LIVE)
    else:
        !python -u /content/pwm/atlas/delhi/tools/fetch_isd_live.py --meta /content/raw/W5K_META.npz --out {LIVE} --years 2024,2025 --end 2025-08-24T23 | tee {W}/small/live_data_log_r25.txt
        os.makedirs(f'{W}/live', exist_ok=True)
        for f in ('W5K_X.npy', 'W5K_OBS.npy', 'W5K_META.npz'): shutil.copy(f'{LIVE}/{f}', f'{W}/live/')
    !ATLAS_OUT={H2} PW_RAW={LIVE} {SC} --year live --origin-every 96 --atlas atlas10m_chk={A10} --models "" 2>&1 | grep --line-buffered -v Warning
    import numpy as np
    a, b = np.load(f'{H2}/h2h/atlas10m_live.npz'), np.load(f'{H2}/h2h/atlas10m_chk_live.npz')
    SAME = (a['cnt'].shape == b['cnt'].shape and (a['cnt'] == b['cnt']).all() and (a['starts'] == b['starts']).all()
            and np.allclose(a['sae'], b['sae'], rtol=1e-3, atol=1e-3))
    print('LIVE DATA IDENTICAL TO PHASE 5b:', SAME)
    if not SAME:      # pairing would be invalid: keep the old files aside and re-score every earlier model on the rebuilt data
        old = f'{H2}/h2h/live_phase5b'; os.makedirs(old, exist_ok=True)
        for f in glob.glob(f'{H2}/h2h/*_live.npz') + [f'{H2}/h2h/h2h_live.json']:
            if os.path.exists(f) and 'chk' not in f: shutil.move(f, old)
        !ATLAS_OUT={H2} PW_RAW={LIVE} {SC} --year live --origin-every 96 --atlas atlas10m={A10} --models persist,chronos-bolt,chronos2,chronos2mv,tfm25,tfm3 2>&1 | grep --line-buffered -v Warning
    !ATLAS_OUT={H2} PW_RAW={LIVE} {SC} --year live --origin-every 96 --atlas atlas25m={A25} --models "" 2>&1 | grep --line-buffered -v Warning


In [ ]:
# 7. The new opponents, zero-shot, on the 2023 test and the live period (same windows, 512 h of history, median scored):
#    TTM r2 (IBM Research), TiRex (NX-AI), Sundial base (Tsinghua THUML), Time-MoE 200M. Each runs on its own; a model that
#    fails to load is reported, not retried with other settings.
if DONE:
    !pip install -q --no-deps granite-tsfm tirex-ts && pip install -q einops deprecated 2>&1 | tail -1
    !pip install -q uv && uv venv -q --system-site-packages /content/tf440 && uv pip install -q --python /content/tf440/bin/python "transformers==4.40.1" 2>&1 | tail -1
    for yr, oe, raw in (('test', 48, '/content/raw'), ('live', 96, LIVE)):
        for m in ('ttm', 'tirex'):
            !ATLAS_OUT={H2} PW_RAW={raw} {SC} --year {yr} --origin-every {oe} --models {m} 2>&1 | grep --line-buffered -v Warning | tail -4
        for m in ('sundial', 'timemoe200'):
            !ATLAS_OUT={H2} PW_RAW={raw} /content/tf440/bin/{SC} --year {yr} --origin-every {oe} --models {m} 2>&1 | grep --line-buffered -v Warning | tail -4


In [ ]:
# 8. Paired tests (station-block bootstrap 10,000 + Diebold-Mariano, Holm over every pair of the family, alpha 0.01)
OPP = ['persist', 'chronos-bolt', 'chronos2', 'chronos2mv', 'tfm25', 'tfm3', 'ttm', 'tirex', 'sundial', 'timemoe200']
if DONE:
    for yr in ('test', 'live'):
        have = [k for k in OPP if os.path.exists(f'{H2}/h2h/{k}_{yr}.npz')]
        print(f'\n===== {yr}: opponents scored: {have}; missing: {[k for k in OPP if k not in have]}')
        for ours in ('atlas25m', 'atlas10m'):
            !ATLAS_OUT={H2} python paired_w5k.py --year {yr} --ours {ours} --vs {','.join(have)} --suffix _r25
        !ATLAS_OUT={H2} python paired_w5k.py --year {yr} --ours atlas25m --vs atlas10m --suffix _vs10m
    !ATLAS_OUT={H2} python paired_w5k.py --year val --ours atlas25m --vs atlas10m --suffix _vs10m


In [ ]:
# 9. Summary
if DONE:
    r = json.load(open(RES25))
    print('25M:', f"{r['params']:,}", 'parameters | GPU', r.get('gpu'), '| step-0 val', r.get('val_step0'), '| val E', r['val_E'],
          '| adopted', r.get('adopted'), '| train min', r.get('minutes_train'))
    print('2023 test (all stations, every hour) MAE:', r.get('test', {}).get('MAE'))
    for yr in ('val', 'test', 'live'):
        f = f'{H2}/h2h/h2h_{yr}.json'
        if os.path.exists(f):
            t = json.load(open(f)); print(f'\n{yr}:')
            for k, v in sorted(((k, v) for k, v in t.items() if not k.startswith('_')), key=lambda kv: kv[1]['E']):
                print(f"  {k:<14} E {v['E']:.5f}  " + '  '.join(f'{x:.4f}' for x in v['MAE'].values()))
